## Example 1: a hook specification, plugins that implement it, and a hook call

In [ ]:
# Example 1: a hook specification, two plugins that implement it, and a call through the plugin manager.
import importlib.metadata

import pluggy

print(f"pluggy {importlib.metadata.version('pluggy')}")

# Markers are project-scoped: the name given to HookspecMarker / HookimplMarker must match the
# PluginManager's project name, or the implementations do not bind to the specification.
hookspec = pluggy.HookspecMarker("demo")
hookimpl = pluggy.HookimplMarker("demo")


class Spec:
    @hookspec
    def process_data(self, data):
        """Process input data."""


class Upper:
    @hookimpl
    def process_data(self, data):
        return data.upper()


class Length:
    @hookimpl
    def process_data(self, data):
        return len(data)


pm = pluggy.PluginManager("demo")
pm.add_hookspecs(Spec)
pm.register(Upper(), name="upper")
pm.register(Length(), name="length")

# Hooks are called with keyword arguments and return one result per implementation.
results = pm.hook.process_data(data="hello world")
names = sorted(name for name, _ in pm.list_name_plugin())
print(f"plugins: {names}")
print(f"results: {results}")
assert names == ["length", "upper"], names
assert pm.has_plugin("upper") and isinstance(pm.get_plugin("length"), Length)
# The most recently registered plugin is called first.
assert results == [11, "HELLO WORLD"], results

## Example 2: `firstresult` hooks

In [ ]:
# Example 2: a `firstresult` hook returns the first non-None result instead of a list.
import pluggy

hookspec = pluggy.HookspecMarker("demo-firstresult")
hookimpl = pluggy.HookimplMarker("demo-firstresult")


class Spec:
    @hookspec(firstresult=True)
    def find_handler(self, name):
        """Return a handler for `name`, or None to let the next plugin answer."""


class Fallback:
    @hookimpl
    def find_handler(self, name):
        return f"fallback:{name}"


class CsvOnly:
    @hookimpl
    def find_handler(self, name):
        return "csv-handler" if name == "csv" else None


pm = pluggy.PluginManager("demo-firstresult")
pm.add_hookspecs(Spec)
pm.register(Fallback())
pm.register(CsvOnly())  # registered last, so asked first

csv_handler = pm.hook.find_handler(name="csv")
json_handler = pm.hook.find_handler(name="json")
print(f"csv -> {csv_handler!r}, json -> {json_handler!r}")
assert csv_handler == "csv-handler", csv_handler
# CsvOnly returns None for anything else, so the call moves on to Fallback.
assert json_handler == "fallback:json", json_handler

## Example 3: call order with `tryfirst` and `trylast`

In [ ]:
# Example 3: call order: `tryfirst` implementations run first, `trylast` ones last, and the rest in
# reverse registration order.
import pluggy

hookspec = pluggy.HookspecMarker("demo-order")
hookimpl = pluggy.HookimplMarker("demo-order")


class Spec:
    @hookspec
    def setup(self, calls):
        """Append this plugin's label to `calls`."""


def make_plugin(label, **marker_options):
    class Plugin:
        @hookimpl(**marker_options)
        def setup(self, calls):
            calls.append(label)

    return Plugin()


pm = pluggy.PluginManager("demo-order")
pm.add_hookspecs(Spec)
pm.register(make_plugin("first-registered"))
pm.register(make_plugin("trylast", trylast=True))
pm.register(make_plugin("tryfirst", tryfirst=True))
pm.register(make_plugin("last-registered"))

calls = []
results = pm.hook.setup(calls=calls)
print(f"call order: {calls}")
assert calls == ["tryfirst", "last-registered", "first-registered", "trylast"], calls
# Every implementation returned None, and None results are left out of the list.
assert results == [], results

## Example 4: hook wrappers

In [ ]:
# Example 4: a hook wrapper runs around the other implementations and can change their result.
import pluggy

hookspec = pluggy.HookspecMarker("demo-wrapper")
hookimpl = pluggy.HookimplMarker("demo-wrapper")


class Spec:
    @hookspec
    def collect(self, items):
        """Return something derived from `items`."""


class Doubler:
    @hookimpl
    def collect(self, items):
        return [item * 2 for item in items]


class Wrapper:
    # A `wrapper=True` implementation is a generator: the code before `yield` runs before the other
    # implementations, `yield` returns their result list, and the wrapper's return value replaces it.
    @hookimpl(wrapper=True)
    def collect(self, items):
        events.append("before")
        result = yield
        events.append("after")
        return result + ["added-by-wrapper"]


events = []
pm = pluggy.PluginManager("demo-wrapper")
pm.add_hookspecs(Spec)
pm.register(Wrapper())
pm.register(Doubler())

out = pm.hook.collect(items=[1, 2])
print(f"wrapped result: {out}")
assert out == [[2, 4], "added-by-wrapper"], out
assert events == ["before", "after"], events

## Example 5: argument validation, unregistering and historic hooks

In [ ]:
# Example 5: argument validation at registration, unregistering, and historic hooks.
import pluggy

hookspec = pluggy.HookspecMarker("demo-lifecycle")
hookimpl = pluggy.HookimplMarker("demo-lifecycle")


class Spec:
    @hookspec
    def greet(self, name):
        """Return a greeting."""

    @hookspec(historic=True)
    def configure(self, config):
        """Receive the configuration, also when the plugin is registered after the call."""


class Hello:
    @hookimpl
    def greet(self, name):
        return f"hello {name}"


class AsksForUnknownArgument:
    @hookimpl
    def greet(self, name, unknown_argument):
        return "never called"


pm = pluggy.PluginManager("demo-lifecycle")
pm.add_hookspecs(Spec)

# An implementation that asks for an argument the specification does not define is refused.
try:
    pm.register(AsksForUnknownArgument())
except pluggy.PluginValidationError as exc:
    print(f"refused at registration: {type(exc).__name__}")
else:
    raise AssertionError("an implementation with an unknown argument was accepted")

hello = Hello()
pm.register(hello)
assert pm.is_registered(hello)
assert pm.hook.greet(name="pluggy") == ["hello pluggy"]
pm.unregister(hello)
assert not pm.is_registered(hello)
assert pm.hook.greet(name="pluggy") == []

# A historic call is remembered and replayed to every plugin registered afterwards.
received = []


class Configurable:
    @hookimpl
    def configure(self, config):
        received.append(config["level"])


pm.hook.configure.call_historic(kwargs={"config": {"level": "debug"}})
assert received == [], received
pm.register(Configurable())
assert received == ["debug"], received
print(f"historic call replayed to a late plugin: {received}")